In [ ]:
import ast
import pandas as pd
from jinja2 import Template

df = pd.read_csv("original/rollouts_original.csv")

problem_idx = [(p, idx) for idx, p in enumerate(list(df["Problem"].unique()))]

df["Problem Index"] = df["Problem"].apply(lambda x: [idx for p, idx in problem_idx if p == x][0])

accepted_df = df.loc[~(df["Judge Decisions"].str.contains("REJECT"))]
rejected_df = df.loc[df["Judge Decisions"].str.contains("REJECT")]

accepted_df["Conversation"] = accepted_df["Conversation"].apply(ast.literal_eval)
accepted_df["Judge Decisions"] = accepted_df["Judge Decisions"].apply(ast.literal_eval)
accepted_df["Conversation from student perspective"] = accepted_df["Conversation from student perspective"].apply(ast.literal_eval)
rejected_df["Judge Decisions"] = rejected_df["Judge Decisions"].apply(ast.literal_eval)
rejected_df["Conversation"] = rejected_df["Conversation"].apply(ast.literal_eval)
rejected_df["Conversation from student perspective"] = rejected_df["Conversation from student perspective"].apply(ast.literal_eval)

def student_to_teacher_perspective(problem, student_name, conversation):
    teacher_conversation = []
    if pd.isnull(student_name):
        student_name = None
    system_prompt = Template(
        open("../prompt_templates/teacher_prompt.txt").read()
    ).render(problem=problem, student_name=student_name)
    teacher_conversation.append({"role": "system", "message": system_prompt})
    
    for i, msg in enumerate(conversation):
        if msg['role'] == "student":
            teacher_conversation.append({"role": "user", "message": msg['content']})
        else:
            teacher_conversation.append({"role": "assistant", "message": msg['content']})
    return teacher_conversation

accepted_df["Conversation from teacher perspective"] = accepted_df.apply(lambda row: student_to_teacher_perspective(row["Problem"], row["Student Name"], row["Conversation"]), axis=1)
rejected_df["Conversation from teacher perspective"] = rejected_df.apply(lambda row: student_to_teacher_perspective(row["Problem"], row["Student Name"], row["Conversation"]), axis=1)

/tmp/ipykernel_841523/3479729369.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  accepted_df["Conversation"] = accepted_df["Conversation"].apply(ast.literal_eval)
/tmp/ipykernel_841523/3479729369.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  accepted_df["Judge Decisions"] = accepted_df["Judge Decisions"].apply(ast.literal_eval)
/tmp/ipykernel_841523/3479729369.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer

In [45]:
accepted_df.head()

,State,Problem,Answer,Conversation,Type,Student Persona,Student Name,Judge Decisions,Solutions,Rewards,Initial Attempts,Initial Rewards,Conversation from student perspective,Problem Index,Conversation from teacher perspective
3,END,Joe bikes $x$ miles East at $20$ mph to hi...,10.0,"[{'role': 'teacher', 'content': 'Sure! Let's s...",GUIDED,simple_student,Alex,{'does_not_leak_answer': [{'reasoning': 'The t...,"[""**Problem:**\n\nJoe bikes $x$ miles East at ...","[0.0, 0.0, 0.0, 0.0]",[],[],"[{'role': 'user', 'content': 'Sure! Let's star...",0,"[{'role': 'system', 'message': ' You are taske..."
9,END,If $(2x+4)^{2010} = a_0 + a_1x + a_2x^2 + \ldo...,0.0,"[{'role': 'student', 'content': 'Here is my at...",ATTEMPTED,simple_student,Riley,{'does_not_leak_answer': [{'reasoning': 'The t...,"[""**Problem:**\nFind the remainder when $a_0 +...","[0.0, 0.0, 0.0, 0.0]",[],[],"[{'role': 'assistant', 'content': 'Here is my ...",1,"[{'role': 'system', 'message': ' You are taske..."
11,END,If $(2x+4)^{2010} = a_0 + a_1x + a_2x^2 + \ldo...,0.0,"[{'role': 'student', 'content': 'Here is my at...",ATTEMPTED,simple_student,Riley,{'does_not_leak_answer': [{'reasoning': 'The t...,['**Problem 1: \\((3x + 2)^{10}\\)**\n\nTo fin...,"[0.0, 0.0, 0.0, 0.0]",[],[],"[{'role': 'assistant', 'content': 'Here is my ...",1,"[{'role': 'system', 'message': ' You are taske..."
12,END,If $(2x+4)^{2010} = a_0 + a_1x + a_2x^2 + \ldo...,0.0,"[{'role': 'student', 'content': 'Here is my at...",ATTEMPTED,simple_student,Riley,{'does_not_leak_answer': [{'reasoning': 'The t...,['**Complete Solution**\n\n**Problem 4**\n\nGi...,"[0.0, 0.0, 0.0, 0.0]",[],[],"[{'role': 'assistant', 'content': 'Here is my ...",1,"[{'role': 'system', 'message': ' You are taske..."
14,END,If $(2x+4)^{2010} = a_0 + a_1x + a_2x^2 + \ldo...,0.0,"[{'role': 'student', 'content': 'Here is my at...",ATTEMPTED,simple_student,Riley,{'does_not_leak_answer': [{'reasoning': 'The t...,"[""**Problem:**\nGiven the polynomial \\(P(x) =...","[1.0, 1.0, 1.0, 1.0]",[],[],"[{'role': 'assistant', 'content': 'Here is my ...",1,"[{'role': 'system', 'message': ' You are taske..."


In [46]:
rejected_df.head()

,State,Problem,Answer,Conversation,Type,Student Persona,Student Name,Judge Decisions,Solutions,Rewards,Initial Attempts,Initial Rewards,Conversation from student perspective,Problem Index,Conversation from teacher perspective
0,END,Joe bikes $x$ miles East at $20$ mph to hi...,10.0,"[{'role': 'teacher', 'content': 'Sure! Let's b...",GUIDED,simple_student,Alex,{'does_not_leak_answer': [{'reasoning': 'The t...,[],[],[],[],"[{'role': 'user', 'content': 'Sure! Let's brea...",0,"[{'role': 'system', 'message': ' You are taske..."
1,END,Joe bikes $x$ miles East at $20$ mph to hi...,10.0,"[{'role': 'teacher', 'content': 'Sure, let's b...",GUIDED,simple_student,Alex,{'does_not_leak_answer': [{'reasoning': 'The t...,[],[],[],[],"[{'role': 'user', 'content': 'Sure, let's brea...",0,"[{'role': 'system', 'message': ' You are taske..."
2,END,Joe bikes $x$ miles East at $20$ mph to hi...,10.0,"[{'role': 'teacher', 'content': 'Let's start b...",GUIDED,simple_student,Alex,{'does_not_leak_answer': [{'reasoning': 'The t...,[],[],[],[],"[{'role': 'user', 'content': 'Let's start by b...",0,"[{'role': 'system', 'message': ' You are taske..."
4,END,Joe bikes $x$ miles East at $20$ mph to hi...,10.0,"[{'role': 'teacher', 'content': 'Sure! Let's b...",GUIDED,simple_student,Alex,{'does_not_leak_answer': [{'reasoning': 'The t...,[],[],[],[],"[{'role': 'user', 'content': 'Sure! Let's brea...",0,"[{'role': 'system', 'message': ' You are taske..."
5,END,Joe bikes $x$ miles East at $20$ mph to hi...,10.0,"[{'role': 'teacher', 'content': 'Sure, Alex! L...",GUIDED,simple_student,Alex,{'does_not_leak_answer': [{'reasoning': 'The t...,[],[],[],[],"[{'role': 'user', 'content': 'Sure, Alex! Let'...",0,"[{'role': 'system', 'message': ' You are taske..."


In [47]:
accepted_df.loc[accepted_df["Student Name"].isnull()]["Conversation from teacher perspective"].iloc[0]

[{'role': 'system',
  'message': '\nYou are tasked with being a teacher and helping a student with a math problem.\n\n\nYou must not reveal the answer to the problem to the student at any point in time.\nYour task is to guide the student to have a complete understanding of the problem.\nEven if the student is already able to solve the problem, you should help them understand and improve the solution so that they get as high of a grade as possible.\n\nIf possible, do not respond with overly long responses to the student.\n\n\n\nYou can end a conversation by writing <end_of_conversation>, please try to end conversations as soon as they are finished instead of prolonging them if not needed. But do not end them prematurely either.\n\nHere is the math problem:\nThis year in October, our school received a donation of footballs, basketballs, and volleyballs totaling 20 balls from a health and pharmaceutical company, with a total value of 330 yuan. The prices of these balls are 60 yuan for eac

In [56]:
accepted_df["Conversation from teacher perspective"].iloc[0]

[{'role': 'system',
  'message': '\nYou are tasked with being a teacher and helping a student named Alex with a math problem.\n\n\nYou must not reveal the answer to the problem to the student at any point in time.\nYour task is to guide the student to have a complete understanding of the problem.\nEven if the student is already able to solve the problem, you should help them understand and improve the solution so that they get as high of a grade as possible.\n\nIf possible, do not respond with overly long responses to the student.\n\n\n\nYou can end a conversation by writing <end_of_conversation>, please try to end conversations as soon as they are finished instead of prolonging them if not needed. But do not end them prematurely either.\n\nHere is the math problem:\nJoe bikes  $x$  miles East at  $20$  mph to his friend’s house. He then turns South and bikes  $x$  miles at  $20$  mph to the store. Then, Joe turns East again and goes to his grandma’s house at  $14$  mph. On this last l

In [ ]:
preference_dict = {
    "accepted": accepted_df["Conversation from teacher perspective"].tolist(),
    "accepted_problem_idx": accepted_df["Problem Index"].tolist(),
    "rejected": rejected_df["Conversation from teacher perspective"].tolist(),
    "rejected_problem_idx": rejected_df["Problem Index"].tolist()
}

import json

with open("preference.json", "w") as f:
    json.dump(preference_dict, f)